In [1]:
from pathlib import Path
import os
import re
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
from adjustText import adjust_text

from scipy.sparse import issparse
import scipy.sparse
from sklearn.metrics import roc_auc_score

from llm_sc_curator import LLMscCurator
from llm_sc_curator.backends import BaseLLMBackend
from llm_sc_curator.noise_lists import NOISE_PATTERNS, NOISE_LISTS
from benchmarks.gt_mappings import get_cd8_ground_truth

warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# 0. Reproducibility / paths
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

BASE = Path("/work")
INPUT_DIR = BASE / "paper" / "gb_resubmission" / "input"
OUTPUT_DIR = BASE / "paper" / "gb_resubmission" / "output" / "Fig2a_d_FigS1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = INPUT_DIR / "cd8_benchmark_data.h5ad"

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

# =============================================================================
# Fig. 1 / problem-schematic settings
# =============================================================================
FIG1_GINI_CLUSTER = "CD8.c01.Tn.MAL"   # fixed visual exemplar for Gini / rank-shift
FIG1_TOPN_GRID = [10, 20, 30, 50, 100, 200]
FIG1_SELECT_N = 50

# Main Fig.1 uses an illustrative exemplar, not the objectively most-burdened cluster
FORCE_FIG1_PROBLEM_CLUSTER = "CD8.c01.Tn.MAL"

print("FIG1_GINI_CLUSTER:", FIG1_GINI_CLUSTER)
print("FIG1_SELECT_N:", FIG1_SELECT_N)
print("FORCE_FIG1_PROBLEM_CLUSTER:", FORCE_FIG1_PROBLEM_CLUSTER)

DATA_PATH: /work/paper/gb_resubmission/input/cd8_benchmark_data.h5ad
OUTPUT_DIR: /work/paper/gb_resubmission/output/Fig2a_d_FigS1
FIG1_GINI_CLUSTER: CD8.c01.Tn.MAL
FIG1_SELECT_N: 50
FORCE_FIG1_PROBLEM_CLUSTER: CD8.c01.Tn.MAL


In [3]:
# =============================================================================
# 1. Load data
# =============================================================================
adata = sc.read_h5ad(DATA_PATH)
assert "meta.cluster" in adata.obs.columns, "meta.cluster not found in adata.obs"
assert "counts" in adata.layers, 'adata.layers["counts"] not found'

adata.obs["Ground_Truth"] = adata.obs["meta.cluster"].apply(get_cd8_ground_truth)

if "highly_variable" not in adata.var.columns:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=2000,
        flavor="seurat_v3",
        layer="counts",
        subset=False,
    )
    print("Computed HVGs from counts")
else:
    print("HVGs already present")

print(adata)
print(adata.obs["Ground_Truth"].value_counts())


Computed HVGs from counts
AnnData object with n_obs × n_vars = 4466 × 24148
    obs: 'cancerType', 'patient', 'libraryID', 'loc', 'meta.cluster', 'platform', 'Cancer_Type', 'Sample_ID', 'Ground_Truth'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'log1p', 'hvg'
    layers: 'counts', 'logcounts'
Ground_Truth
CD8_EffectorMemory    1983
CD8_Exhausted          928
CD8_Effector           753
CD8_Naive              300
CD8_MAIT               300
CD8_ISG                202
Name: count, dtype: int64


In [4]:
# =============================================================================
# Cluster color palette for supplementary figures
# =============================================================================
def make_cluster_color_map(cluster_names):
    """
    Assign a stable color to each cluster.
    Uses tab20 first, then falls back to hsv if needed.
    """
    cluster_names = list(sorted(map(str, cluster_names)))
    n = len(cluster_names)

    if n <= 20:
        cmap = plt.get_cmap("tab20")
        colors = [mpl.colors.to_hex(cmap(i)) for i in range(n)]
    else:
        cmap = plt.get_cmap("hsv")
        colors = [mpl.colors.to_hex(cmap(i / n)) for i in range(n)]

    return dict(zip(cluster_names, colors))

ALL_CLUSTER_NAMES = sorted(adata.obs["meta.cluster"].astype(str).unique())
CLUSTER_COLOR_MAP = make_cluster_color_map(ALL_CLUSTER_NAMES)

print("Number of clusters in color map:", len(CLUSTER_COLOR_MAP))
print(list(CLUSTER_COLOR_MAP.items())[:5])

Number of clusters in color map: 17
[('CD8.c01.Tn.MAL', '#1f77b4'), ('CD8.c02.Tm.IL7R', '#aec7e8'), ('CD8.c03.Tm.RPS12', '#ff7f0e'), ('CD8.c04.Tm.CD52', '#ffbb78'), ('CD8.c05.Tem.CXCR5', '#2ca02c')]


In [5]:
# =============================================================================
# 2. Null backend for truly backend-free evaluation
# =============================================================================
class NullBackend(BaseLLMBackend):
    def generate(self, prompt: str, json_mode: bool = False) -> str:
        raise RuntimeError("NullBackend does not support annotation calls.")

curator = LLMscCurator(backend=NullBackend())
curator.set_global_context(adata)


In [6]:
# =============================================================================
# 3. Noise definitions
# =============================================================================
COMPILED_REGEX_NOISE = {
    name: re.compile(pattern)
    for name, pattern in NOISE_PATTERNS.items()
}

CURATED_NOISE_GENE_SET = set()
for _, genes in NOISE_LISTS.items():
    CURATED_NOISE_GENE_SET.update(genes)

def is_regex_noise_gene(gene: str) -> bool:
    g = str(gene)
    return any(p.search(g) for p in COMPILED_REGEX_NOISE.values())

def is_any_noise_gene(gene: str) -> bool:
    g = str(gene)
    if g in CURATED_NOISE_GENE_SET:
        return True
    return is_regex_noise_gene(g)

In [7]:
# =============================================================================
# 4. # Shared canonical non-identity taxonomy
#   - used by Fig. 1
#   - used by Supplementary PDF
#   - used by reason-to-category collapsing
# =============================================================================

COMPILED_REGEX_BY_MODULE = {
    name: re.compile(pattern)
    for name, pattern in NOISE_PATTERNS.items()
}

CURATED_LIST_BY_MODULE = {
    name: set(map(str, genes))
    for name, genes in NOISE_LISTS.items()
}

# Display order (bottom -> top)
CANONICAL_NONIDENTITY_CATEGORY_ORDER = [
    "Housekeeping (Low Gini)",
    "Uninformative IDs / predicted loci",
    "State Noise (Cell cycle / stress / IFN / MHC / sex)",
    "Ubiquitous Program (Translation / Histone)",
    "Hemo / contamination flag",
    "Tech Noise (Mito/Ribo)",
    "Lineage Noise (TCR/Ig)",
    "Other / uncategorized noise",
]

CANONICAL_NONIDENTITY_COLOR_MAP = {
    "Housekeeping (Low Gini)": "#F2B134",
    "Uninformative IDs / predicted loci": "#90A4AE",
    "State Noise (Cell cycle / stress / IFN / MHC / sex)": "#66BB6A",
    "Ubiquitous Program (Translation / Histone)": "#5C6BC0",
    "Hemo / contamination flag": "#8D6E63",
    "Tech Noise (Mito/Ribo)": "#EF5350",
    "Lineage Noise (TCR/Ig)": "#AB47BC",
    "Other / uncategorized noise": "#78909C",
}

# Priority for assigning a single category when multiple modules match
CANONICAL_NONIDENTITY_PRIORITY = [
    "Lineage Noise (TCR/Ig)",
    "Tech Noise (Mito/Ribo)",
    "Hemo / contamination flag",
    "State Noise (Cell cycle / stress / IFN / MHC / sex)",
    "Ubiquitous Program (Translation / Histone)",
    "Uninformative IDs / predicted loci",
    "Housekeeping (Low Gini)",
    "Other / uncategorized noise",
]

# Backward-compatible aliases for the rest of the notebook/script
FIG1_NONIDENTITY_CATEGORY_ORDER = CANONICAL_NONIDENTITY_CATEGORY_ORDER
FIG1_NONIDENTITY_COLOR_MAP = CANONICAL_NONIDENTITY_COLOR_MAP
FIG1_NONIDENTITY_PRIORITY = CANONICAL_NONIDENTITY_PRIORITY


def _collapse_noise_module_name(module_name: str) -> str:
    """
    Collapse package-level noise modules into the shared canonical categories.
    """
    m = str(module_name).lower()

    # lineage / clonotype
    if any(k in m for k in ["tcr", "bcr", "ig", "vdj"]):
        return "Lineage Noise (TCR/Ig)"

    # mito / ribo
    if any(k in m for k in ["mito", "ribo", "ribosomal"]):
        return "Tech Noise (Mito/Ribo)"

    # hemoglobin / contamination
    if "hemo" in m or "hemoglobin" in m:
        return "Hemo / contamination flag"

    # cell cycle / proliferation / stress / IFN / MHC / sex
    if any(k in m for k in [
        "cellcycle", "cell_cycle", "cycling", "g1s", "g2m", "prolif",
        "stress", "junfos", "heatshock", "hla", "mhc", "interferon", "sexchromosome"
    ]):
        return "State Noise (Cell cycle / stress / IFN / MHC / sex)"

    # uninformative IDs / predicted loci
    if any(k in m for k in ["ensembl", "linc", "gm", "rik", "loc"]):
        return "Uninformative IDs / predicted loci"

    # translation / histone
    if any(k in m for k in ["translation", "histone"]):
        return "Ubiquitous Program (Translation / Histone)"

    return "Other / uncategorized noise"


def simplify_reason_to_canonical_category(reason: str) -> str:
    """
    Map masker reason strings to the same canonical categories used in Fig. 1.
    """
    r = str(reason).strip()
    rl = r.lower()

    if r in {"None", "", "nan"}:
        return "Kept"

    if "low_gini" in rl:
        return "Housekeeping (Low Gini)"

    if any(k in rl for k in ["ribo", "mito", "ribosomal"]):
        return "Tech Noise (Mito/Ribo)"

    if any(k in rl for k in ["tcr", "bcr", "ig", "vdj"]):
        return "Lineage Noise (TCR/Ig)"

    if any(k in rl for k in ["hemo", "hemoglobin"]):
        return "Hemo / contamination flag"

    if any(k in rl for k in [
        "cellcycle", "cell_cycle", "cycling", "g1s", "g2m", "prolif",
        "stress", "junfos", "heatshock", "interferon", "hla", "mhc", "sexchromosome"
    ]):
        return "State Noise (Cell cycle / stress / IFN / MHC / sex)"

    if any(k in rl for k in ["translation", "histone"]):
        return "Ubiquitous Program (Translation / Histone)"

    if any(k in rl for k in ["ensembl", "linc", "loc", "gm", "rik"]):
        return "Uninformative IDs / predicted loci"

    return "Other / uncategorized noise"


def _matched_noise_modules(gene: str):
    """
    Return all matching package-level noise modules for a gene.
    """
    g = str(gene)
    hits = []

    for name, pat in COMPILED_REGEX_BY_MODULE.items():
        try:
            if pat.search(g):
                hits.append(name)
        except Exception:
            pass

    for name, gene_set in CURATED_LIST_BY_MODULE.items():
        if g in gene_set:
            hits.append(name)

    out = []
    seen = set()
    for h in hits:
        if h not in seen:
            out.append(h)
            seen.add(h)
    return out


def is_low_gini_housekeeping_gene(gene: str, gene_stats=None, low_gini_thr=None, mean_floor=0.01) -> bool:
    """
    Orthogonal non-identity burden: globally broad / housekeeping-like genes.
    """
    if gene_stats is None or low_gini_thr is None:
        return False

    g = str(gene)
    if g not in gene_stats.index:
        return False

    gini_val = gene_stats.at[g, "gini"] if "gini" in gene_stats.columns else np.nan
    mean_val = gene_stats.at[g, "mean"] if "mean" in gene_stats.columns else np.nan

    if pd.isna(gini_val) or pd.isna(mean_val):
        return False

    return bool((mean_val >= mean_floor) and (gini_val <= low_gini_thr))


def classify_non_identity_category_for_fig1(gene: str, gene_stats=None, low_gini_thr=None) -> str:
    """
    Single-category assignment for Fig. 1 problem schematic.
    Returns one of FIG1_NONIDENTITY_CATEGORY_ORDER or 'Non-noise'.
    """
    g = str(gene)
    g_upper = g.upper()

    modules = _matched_noise_modules(g)
    if len(modules) > 0:
        cats = [_collapse_noise_module_name(m) for m in modules]
        for cat in FIG1_NONIDENTITY_PRIORITY:
            if cat in cats:
                return cat

    # fallback heuristics for robustness
    if re.match(r"^(TRA|TRB|TRD|TRG|IGH|IGK|IGL)", g_upper):
        return "Lineage Noise (TCR/Ig)"
    if g_upper.startswith("MT-") or re.match(r"^(RPL|RPS)", g_upper):
        return "Tech Noise (Mito/Ribo)"
    if re.match(r"^(EIF|EEF)", g_upper) or re.match(r"^(HIST)", g_upper):
        return "Ubiquitous Program (Translation / Histone)"
    if re.match(r"^(HLA-[ABCEFG]|H2-[DKL]|XIST|UTY|DDX3Y)$", g_upper):
        return "State Noise (Cell cycle / stress / IFN / MHC / sex)"
    if re.match(r"^(ENSG|ENSMUSG|LOC\d+|GM\d+)", g_upper):
        return "Uninformative IDs / predicted loci"
    if re.match(r"^(HBA|HBB)", g_upper):
        return "Hemo / contamination flag"

    if is_low_gini_housekeeping_gene(g, gene_stats=gene_stats, low_gini_thr=low_gini_thr):
        return "Housekeeping (Low Gini)"

    return "Non-noise"


def is_non_identity_gene_for_fig1(gene: str, gene_stats=None, low_gini_thr=None) -> bool:
    return classify_non_identity_category_for_fig1(
        gene,
        gene_stats=gene_stats,
        low_gini_thr=low_gini_thr
    ) != "Non-noise"

def get_standard_hvg_deg_genes(de_df_raw, adata, n_top=50):
    """
    Naive 'standard' input for the Fig. 1 problem schematic:
    ranked DEGs with preferential restriction to HVGs if available.

    This better matches the conceptual Standard path: HVGs -> DEGs -> LLM.
    """
    if "highly_variable" not in adata.var.columns:
        return de_df_raw["names"].head(n_top).astype(str).tolist()

    hvgs = set(
        adata.var_names[
            adata.var["highly_variable"].fillna(False).values
        ].astype(str)
    )

    ranked = de_df_raw["names"].astype(str).tolist()
    hvg_ranked = [g for g in ranked if g in hvgs]

    if len(hvg_ranked) >= n_top:
        return hvg_ranked[:n_top]

    fill = [g for g in ranked if g not in set(hvg_ranked)]
    return (hvg_ranked + fill)[:n_top]

In [8]:
# =============================================================================
# 5. Canonical marker DB for backend-free sanity metrics
# =============================================================================
CD8_MARKER_DB = {
    "CD8_Naive": {
        "IL7R", "CCR7", "LEF1", "TCF7", "SELL", "MAL", "LTB", "KLF2"
    },
    "CD8_EffectorMemory": {
        "GZMK", "LTB", "AQP3", "IL7R", "CXCR4", "ANXA1", "ZFP36L2", "DUSP2"
    },
    "CD8_Effector": {
        "CCL5", "NKG7", "PRF1", "GZMB", "CX3CR1", "KLRG1", "FGFBP2", "GNLY"
    },
    "CD8_Exhausted": {
        "CXCL13", "CTLA4", "TIGIT", "HAVCR2", "PDCD1", "ENTPD1", "TOX", "TNFRSF9"
    },
    "CD8_ISG": {
        "IFIT1", "ISG15", "MX1", "STAT1", "OAS1", "IFI6"
    },
    "CD8_MAIT": {
        "SLC4A10", "KLRB1", "CXCR6"
    },
    "CD8_Cycling": {
        "MKI67", "TOP2A", "CDK1", "BIRC5", "PCNA", "TYMS"
    },
    "CD8_NK_Killer": {
        "NKG7", "GNLY", "PRF1", "GZMB", "FGFBP2"
    },
}


In [9]:
# =============================================================================
# 6. Helpers
# =============================================================================
def _to_1d_array(x):
    if issparse(x):
        return np.asarray(x.toarray()).ravel()
    return np.asarray(x).ravel()

def _get_cluster_masks(adata, cluster_name):
    target_mask = (adata.obs["meta.cluster"].astype(str) == str(cluster_name)).values
    rest_mask = ~target_mask
    return target_mask, rest_mask

def build_de_table(
    adata,
    cluster_name,
    min_target_mean=0.02,
    min_delta_mean=0.02,
    min_logfc=0.2,
    min_target_pct=0.02,
    min_delta_pct=0.02,
):
    """
    Build a DE table with effect-size / abundance statistics, mirroring the
    Stage 1 logic used in LLM-scCurator.
    """
    tmp = "__tmp_binary__"
    adata.obs[tmp] = "Rest"
    adata.obs.loc[adata.obs["meta.cluster"].astype(str) == str(cluster_name), tmp] = "Target"

    sc.tl.rank_genes_groups(
        adata,
        groupby=tmp,
        groups=["Target"],
        reference="Rest",
        method="wilcoxon",
        use_raw=False,
    )
    de_df_raw = sc.get.rank_genes_groups_df(adata, group="Target").copy()

    target_mask = adata.obs[tmp] == "Target"
    rest_mask = adata.obs[tmp] == "Rest"

    X = adata.X
    if issparse(X):
        X_target = X[target_mask.values, :]
        X_rest = X[rest_mask.values, :]
        target_mean = np.asarray(X_target.mean(axis=0)).ravel()
        rest_mean = np.asarray(X_rest.mean(axis=0)).ravel()
        target_pct = np.asarray((X_target > 0).mean(axis=0)).ravel()
        rest_pct = np.asarray((X_rest > 0).mean(axis=0)).ravel()
    else:
        X_target = X[target_mask.values, :]
        X_rest = X[rest_mask.values, :]
        target_mean = X_target.mean(axis=0)
        rest_mean = X_rest.mean(axis=0)
        target_pct = (X_target > 0).mean(axis=0)
        rest_pct = (X_rest > 0).mean(axis=0)

    expr_stats = pd.DataFrame({
        "names": adata.var_names,
        "target_mean": target_mean,
        "rest_mean": rest_mean,
        "target_pct": target_pct,
        "rest_pct": rest_pct,
    })

    de_df = de_df_raw.merge(expr_stats, on="names", how="left")
    de_df["delta_mean"] = de_df["target_mean"] - de_df["rest_mean"]
    de_df["delta_pct"] = de_df["target_pct"] - de_df["rest_pct"]

    eff_mask = (
        (de_df["target_mean"] >= min_target_mean) &
        (de_df["delta_mean"] >= min_delta_mean) &
        (de_df["target_pct"] >= min_target_pct) &
        (de_df["delta_pct"] >= min_delta_pct)
    )
    if "logfoldchanges" in de_df.columns:
        eff_mask &= (de_df["logfoldchanges"].fillna(0) >= min_logfc)

    de_df_filtered = de_df.loc[eff_mask].copy()

    adata.obs.drop(columns=[tmp], inplace=True, errors="ignore")
    return de_df_raw, de_df_filtered

def get_standard_genes(de_df_raw, n_top=50):
    return de_df_raw["names"].head(n_top).tolist()

def get_filter_only_genes(de_df_raw, de_df_filtered, n_top=50):
    if de_df_filtered.empty:
        return de_df_raw["names"].head(n_top).tolist()
    return de_df_filtered["names"].head(n_top).tolist()

def get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=50, oversample=300):
    """
    Effect-size-filtered DE list + regex-only masking.
    This is intentionally NOT using curated NOISE_LISTS so that the variant name
    matches its implementation.
    """
    source_df = de_df_filtered if not de_df_filtered.empty else de_df_raw
    candidates = source_df["names"].head(oversample).tolist()
    clean = [g for g in candidates if not is_regex_noise_gene(g)]
    return clean[:n_top]

def get_full_core_genes(curator, adata, cluster_name, n_top=50):
    genes = curator.curate_features(
        adata,
        group_col="meta.cluster",
        target_group=str(cluster_name),
        n_top=n_top,
        use_statistics=True,
    )
    return genes[:n_top]

def noise_fraction(genes):
    if len(genes) == 0:
        return np.nan
    return float(np.mean([is_any_noise_gene(g) for g in genes]))

def marker_recall_at_n(genes, gt_label, adata_var_names):
    truth = CD8_MARKER_DB.get(gt_label, set())
    truth = [g for g in truth if g in set(map(str, adata_var_names))]
    if len(truth) == 0:
        return np.nan
    hit = sum(g in set(genes) for g in truth)
    return hit / len(truth)

def mean_auroc_specificity(adata, cluster_name, genes):
    """
    Mean one-vs-rest AUROC across genes in the input list.
    Higher is better.
    This is a within-benchmark specificity metric, NOT a cross-lineage leakage metric.
    """
    if len(genes) == 0:
        return np.nan

    target_mask, rest_mask = _get_cluster_masks(adata, cluster_name)
    y_true = target_mask.astype(int)

    scores = []
    for g in genes:
        if g not in adata.var_names:
            continue
        gi = adata.var_names.get_loc(g)
        x = _to_1d_array(adata[:, gi].X)

        if np.all(x == x[0]):
            scores.append(0.5)
            continue

        try:
            auc = roc_auc_score(y_true, x)
        except Exception:
            auc = 0.5
        scores.append(auc)

    if len(scores) == 0:
        return np.nan
    return float(np.mean(scores))




In [10]:
# =============================================================================
# Gini helper block
# =============================================================================

def ensure_global_gini_stats(curator, adata, mean_floor=0.01, gini_q_low=0.01, gini_q_high=0.90, low_gini_cap=0.15):
    """
    Ensure curator.masker.gene_stats exists and return:
      - gene_stats dataframe
      - effective low-Gini cutoff (dynamic q, capped)
      - high-Gini cutoff (top q)
    """
    if curator.masker is None:
        curator.set_global_context(adata)

    if getattr(curator.masker, "gene_stats", None) is None:
        curator.masker.calculate_gene_stats()

    gs = curator.masker.gene_stats.copy()
    if "gini" not in gs.columns or "mean" not in gs.columns:
        raise ValueError("gene_stats must contain 'gini' and 'mean' columns.")

    gs = gs.replace([np.inf, -np.inf], np.nan)
    gs = gs.dropna(subset=["gini", "mean"])

    valid_low = gs.loc[gs["mean"] >= mean_floor, "gini"].dropna()
    if len(valid_low) == 0:
        low_thr = low_gini_cap
    else:
        low_thr = min(np.quantile(valid_low, gini_q_low), low_gini_cap)

    valid_high = gs["gini"].dropna()
    if len(valid_high) == 0:
        high_thr = np.nan
    else:
        high_thr = np.quantile(valid_high, gini_q_high)

    return gs, low_thr, high_thr


def mean_gene_gini(genes, gene_stats):
    vals = [
        float(gene_stats.at[g, "gini"])
        for g in genes
        if g in gene_stats.index and pd.notna(gene_stats.at[g, "gini"])
    ]
    return np.mean(vals) if len(vals) > 0 else np.nan


def low_gini_fraction(genes, gene_stats, low_gini_thr):
    vals = [
        float(gene_stats.at[g, "gini"])
        for g in genes
        if g in gene_stats.index and pd.notna(gene_stats.at[g, "gini"])
    ]
    if len(vals) == 0:
        return np.nan
    return np.mean([v <= low_gini_thr for v in vals])


def high_gini_fraction(genes, gene_stats, high_gini_thr):
    vals = [
        float(gene_stats.at[g, "gini"])
        for g in genes
        if g in gene_stats.index and pd.notna(gene_stats.at[g, "gini"])
    ]
    if len(vals) == 0 or pd.isna(high_gini_thr):
        return np.nan
    return np.mean([v >= high_gini_thr for v in vals])


# prepare once
GLOBAL_GENE_STATS, LOW_GINI_THR, HIGH_GINI_THR = ensure_global_gini_stats(curator, adata)

print("LOW_GINI_THR :", LOW_GINI_THR)
print("HIGH_GINI_THR:", HIGH_GINI_THR)
print(GLOBAL_GENE_STATS.head())

LOW_GINI_THR : 0.15
HIGH_GINI_THR: 0.9993261451703163
                  mean      gini
gene                            
RP11-34P13.3  0.000000  0.000000
RP11-34P13.7  0.000899  0.999265
FO538757.3    0.000000  0.000000
FO538757.2    0.179059  0.878292
AP006222.2    0.000527  0.999178


In [18]:
# =============================================================================
# Supplementary Gini landscape PDF
# =============================================================================
def _get_standard_de_genes_for_cluster(adata, cluster_name, n_top=15):
    tmp = "__tmp_plot_binary__"
    adata.obs[tmp] = "Rest"
    adata.obs.loc[adata.obs["meta.cluster"].astype(str) == str(cluster_name), tmp] = "Target"

    sc.tl.rank_genes_groups(
        adata,
        groupby=tmp,
        groups=["Target"],
        reference="Rest",
        method="wilcoxon",
        use_raw=False,
    )
    df_std = sc.get.rank_genes_groups_df(adata, group="Target").copy()
    adata.obs.drop(columns=[tmp], inplace=True, errors="ignore")

    return get_standard_hvg_deg_genes(df_std, adata, n_top=n_top)


def generate_supplementary_pdf(
    curator,
    adata,
    out_path="Supplementary_Figure_S1_cd8_Cluster_Landscapes.pdf",
    n_label_curated=12,
    n_label_standard=8,
):
    """
    Multi-page PDF of cluster-specific Gini landscapes.

    Key improvements vs earlier version:
    - uses the actual curated feature list from `curate_features(...)`
    - optionally adds a few top standard DE genes for comparison
    - computes the effective low-Gini cutoff dynamically from global gene_stats
    """
    print(f"Generating Supplementary PDF: {out_path} ...")

    global_stats, low_gini_thr, high_gini_thr = ensure_global_gini_stats(curator, adata)

    # use the same biological-noise detection logic as the current curator
    if curator.masker is None:
        curator.set_global_context(adata)

    mask_reasons = curator.masker.detect_biological_noise(
        gini_threshold=None,
        gini_q=0.01,
        mean_floor=0.01,
        whitelist=None,
        rescue_mean_floor=0.05,
        low_gini_cap=0.15,
    )

    plot_base = global_stats.copy()
    plot_base["Status"] = "Kept"
    plot_base["Reason"] = "None"

    for gene, reason in mask_reasons.items():
        if gene in plot_base.index:
            plot_base.loc[gene, "Status"] = "Filtered"
            plot_base.loc[gene, "Reason"] = reason

    plot_base["Category"] = plot_base["Reason"].apply(simplify_reason_to_canonical_category)

    palette = {
        "Kept": "#E0E0E0",
        **CANONICAL_NONIDENTITY_COLOR_MAP,
    }

    with PdfPages(out_path) as pdf:
        unique_clusters = sorted(adata.obs["meta.cluster"].astype(str).unique())

        for i, cluster in enumerate(unique_clusters):
            print(f"  Processing cluster [{i+1}/{len(unique_clusters)}]: {cluster}")

            # signature gene from cluster name
            try:
                signature_gene = cluster.split(".")[-1]
                if signature_gene not in adata.var_names:
                    if signature_gene.upper() in adata.var_names:
                        signature_gene = signature_gene.upper()
                    else:
                        signature_gene = None
            except Exception:
                signature_gene = None

            # local mean for x-axis
            cells = adata[adata.obs["meta.cluster"].astype(str) == cluster]
            if scipy.sparse.issparse(cells.X):
                local_mean = cells.X.mean(axis=0).A1
            else:
                local_mean = np.asarray(cells.X.mean(axis=0)).ravel()

            plot_df = plot_base.copy()
            plot_df["local_mean"] = local_mean
            plot_df["log_local_mean"] = np.log1p(plot_df["local_mean"])
            plot_df.to_csv(OUTPUT_DIR / f"FigS1d_data.{cluster}.csv", index=True)

            # actual curated top genes
            curated_genes = curator.curate_features(
                adata,
                group_col="meta.cluster",
                target_group=cluster,
                n_top=n_label_curated,
                use_statistics=True,
            )

            # comparison: top standard DE genes
            standard_genes = _get_standard_de_genes_for_cluster(
                adata, cluster_name=cluster, n_top=n_label_standard
            )

            # build label set
            label_genes = []
            if signature_gene is not None:
                label_genes.append(signature_gene)

            for g in curated_genes:
                if g not in label_genes:
                    label_genes.append(g)

            for g in standard_genes:
                if g not in label_genes:
                    label_genes.append(g)

            # plotting
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

            fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

            # background
            subset = plot_df[plot_df["Category"] == "Kept"]
            ax.scatter(
                subset["log_local_mean"],
                subset["gini"],
                c="#E0E0E0",
                s=15,
                alpha=0.4,
                edgecolors="none",
            )

            # filtered categories (stable canonical order)
            categories = [
                c for c in CANONICAL_NONIDENTITY_CATEGORY_ORDER
                if c in set(plot_df["Category"]) and c != "Kept"
            ]

            for cat in categories:
                subset = plot_df[plot_df["Category"] == cat]
                color = palette.get(cat, "black")
                ax.scatter(
                    subset["log_local_mean"],
                    subset["gini"],
                    c=color,
                    s=20,
                    alpha=0.8,
                    label=cat,
                    edgecolors="white",
                    linewidth=0.3,
                )

            # thresholds
            ax.axhline(y=low_gini_thr, color="black", linestyle="--", linewidth=1, alpha=0.6)
            if not pd.isna(high_gini_thr):
                ax.axhline(y=high_gini_thr, color="black", linestyle=":", linewidth=1, alpha=0.4)

            texts = []
            curated_set = set(curated_genes)
            standard_set = set(standard_genes)
            

            for gene in label_genes:
                if gene not in plot_df.index:
                    continue

                row = plot_df.loc[gene]

                if gene == signature_gene:
                    t = ax.text(
                        row["log_local_mean"], row["gini"], gene,
                        fontsize=13, fontweight="bold", color="#D32F2F"
                    )
                elif gene in curated_set:
                    t = ax.text(
                        row["log_local_mean"], row["gini"], gene,
                        fontsize=10, fontweight="bold", color="#B71C1C"
                    )
                elif gene in standard_set:
                    t = ax.text(
                        row["log_local_mean"], row["gini"], gene,
                        fontsize=9, fontweight="bold", color="black"
                    )
                else:
                    t = ax.text(
                        row["log_local_mean"], row["gini"], gene,
                        fontsize=8, color="black"
                    )
                texts.append(t)

            adjust_text(
                texts,
                arrowprops=dict(arrowstyle="-", color="black", lw=0.5),
                force_points=0.3,
                force_text=0.5,
                expand_points=(1.2, 1.2),
            )

            ax.set_xlabel(f"Log mean expression ({cluster})", fontsize=14, fontweight="bold")
            ax.set_ylabel("Global Gini coefficient (specificity)", fontsize=14, fontweight="bold")
            #ax.set_title(f"Cluster: {cluster}", fontsize=12, fontweight="bold")
            ax.grid(True, which="major", linestyle=":", linewidth=0.5, alpha=0.5)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ Saved multi-page PDF to: {out_path}")


# --- run ---
generate_supplementary_pdf(
    curator,
    adata,
    out_path=OUTPUT_DIR / "FigS1d.pdf",
)

Generating Supplementary PDF: /work/paper/gb_resubmission/output/Fig2a_d_FigS1/FigS1d.pdf ...
  Processing cluster [1/17]: CD8.c01.Tn.MAL
  Processing cluster [2/17]: CD8.c02.Tm.IL7R
  Processing cluster [3/17]: CD8.c03.Tm.RPS12
  Processing cluster [4/17]: CD8.c04.Tm.CD52
  Processing cluster [5/17]: CD8.c05.Tem.CXCR5
  Processing cluster [6/17]: CD8.c06.Tem.GZMK
  Processing cluster [7/17]: CD8.c07.Temra.CX3CR1
  Processing cluster [8/17]: CD8.c08.Tk.TYROBP
  Processing cluster [9/17]: CD8.c09.Tk.KIR2DL4
  Processing cluster [10/17]: CD8.c10.Trm.ZNF683
  Processing cluster [11/17]: CD8.c11.Tex.PDCD1
  Processing cluster [12/17]: CD8.c12.Tex.CXCL13
  Processing cluster [13/17]: CD8.c13.Tex.myl12a
  Processing cluster [14/17]: CD8.c14.Tex.TCF7
  Processing cluster [15/17]: CD8.c15.ISG.IFIT1
  Processing cluster [16/17]: CD8.c16.MAIT.SLC4A10
  Processing cluster [17/17]: CD8.c17.Tm.NME1
✅ Saved multi-page PDF to: /work/paper/gb_resubmission/output/Fig2a_d_FigS1/FigS1d.pdf


In [12]:
# =============================================================================
# 7. Run backend-free minimal Fig.2 metrics
#    - top50 metrics for panels b/c
#    - top-N recall curve for panel d
#    - NEW: Gini-centered quantitative metrics
# =============================================================================
N_TOP = 50
TOPN_GRID = [10, 20, 30, 50]
EXCLUDE_GT = {"CD8_Other", "Other", "Unknown"}

cluster_list = sorted(adata.obs["meta.cluster"].astype(str).unique())
cluster_list = [c for c in cluster_list if get_cd8_ground_truth(c) not in EXCLUDE_GT]

rows = []
topn_rows = []
gene_rows = []

for i, cluster_name in enumerate(cluster_list, start=1):
    gt = get_cd8_ground_truth(cluster_name)
    print(f"[{i}/{len(cluster_list)}] {cluster_name} -> {gt}")

    de_df_raw, de_df_filtered = build_de_table(adata, cluster_name)

    genes_standard = get_standard_genes(de_df_raw, n_top=N_TOP)
    genes_filter = get_filter_only_genes(de_df_raw, de_df_filtered, n_top=N_TOP)
    genes_regex = get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=N_TOP, oversample=300)
    genes_core = get_full_core_genes(curator, adata, cluster_name, n_top=N_TOP)

    variant_to_genes = {
        "standard": genes_standard,
        "filter_only": genes_filter,
        "regex_mask": genes_regex,
        "full_core": genes_core,
    }

    # ----- top50 metrics (for panels b/c and Gini summaries) -----
    for variant, genes in variant_to_genes.items():
        rows.append({
            "cluster": cluster_name,
            "Ground_Truth": gt,
            "variant": variant,
            "noise_fraction": noise_fraction(genes),
            "marker_recall": marker_recall_at_n(genes, gt, adata.var_names),  # recall@50
            "mean_gene_gini": mean_gene_gini(genes, GLOBAL_GENE_STATS),
            "low_gini_fraction": low_gini_fraction(genes, GLOBAL_GENE_STATS, LOW_GINI_THR),
            "high_gini_fraction": high_gini_fraction(genes, GLOBAL_GENE_STATS, HIGH_GINI_THR),
            "n_genes": len(genes),
        })

        for rank, g in enumerate(genes, start=1):
            gene_rows.append({
                "cluster": cluster_name,
                "Ground_Truth": gt,
                "variant": variant,
                "rank": rank,
                "gene": g,
                "gene_gini": GLOBAL_GENE_STATS.at[g, "gini"] if g in GLOBAL_GENE_STATS.index else np.nan,
                "is_regex_noise": is_regex_noise_gene(g),
                "is_any_noise": is_any_noise_gene(g),
                "is_low_gini": (GLOBAL_GENE_STATS.at[g, "gini"] <= LOW_GINI_THR) if g in GLOBAL_GENE_STATS.index else np.nan,
                "is_high_gini": (GLOBAL_GENE_STATS.at[g, "gini"] >= HIGH_GINI_THR) if g in GLOBAL_GENE_STATS.index else np.nan,
                "is_canonical_marker": g in CD8_MARKER_DB.get(gt, set()),
            })

    # ----- top-N curve data (for panel d + optional Gini curves) -----
    for n_eval in TOPN_GRID:
        for variant, genes in variant_to_genes.items():
            genes_n = genes[:n_eval]
            topn_rows.append({
                "cluster": cluster_name,
                "Ground_Truth": gt,
                "variant": variant,
                "n_top_eval": n_eval,
                "marker_recall_at_n": marker_recall_at_n(genes_n, gt, adata.var_names),
                "noise_fraction_at_n": noise_fraction(genes_n),
                "mean_gene_gini_at_n": mean_gene_gini(genes_n, GLOBAL_GENE_STATS),
                "low_gini_fraction_at_n": low_gini_fraction(genes_n, GLOBAL_GENE_STATS, LOW_GINI_THR),
                "high_gini_fraction_at_n": high_gini_fraction(genes_n, GLOBAL_GENE_STATS, HIGH_GINI_THR),
            })

df_metrics = pd.DataFrame(rows)
df_topn = pd.DataFrame(topn_rows)
df_genes = pd.DataFrame(gene_rows)

df_metrics.to_csv(OUTPUT_DIR / "fig2_cd8_backend_free_metrics.csv", index=False)
df_topn.to_csv(OUTPUT_DIR / "fig2_cd8_backend_free_topn_curve.csv", index=False)
df_genes.to_csv(OUTPUT_DIR / "fig2_cd8_backend_free_gene_lists.csv", index=False)

print("\nHead of df_metrics:")
print(df_metrics.head())

print("\nHead of df_topn:")
print(df_topn.head())

print("\nHead of df_genes:")
print(df_genes.head())

[1/17] CD8.c01.Tn.MAL -> CD8_Naive
[2/17] CD8.c02.Tm.IL7R -> CD8_EffectorMemory
[3/17] CD8.c03.Tm.RPS12 -> CD8_EffectorMemory
[4/17] CD8.c04.Tm.CD52 -> CD8_EffectorMemory
[5/17] CD8.c05.Tem.CXCR5 -> CD8_EffectorMemory
[6/17] CD8.c06.Tem.GZMK -> CD8_EffectorMemory
[7/17] CD8.c07.Temra.CX3CR1 -> CD8_Effector
[8/17] CD8.c08.Tk.TYROBP -> CD8_Effector
[9/17] CD8.c09.Tk.KIR2DL4 -> CD8_Effector
[10/17] CD8.c10.Trm.ZNF683 -> CD8_EffectorMemory
[11/17] CD8.c11.Tex.PDCD1 -> CD8_Exhausted
[12/17] CD8.c12.Tex.CXCL13 -> CD8_Exhausted
[13/17] CD8.c13.Tex.myl12a -> CD8_Exhausted
[14/17] CD8.c14.Tex.TCF7 -> CD8_Exhausted
[15/17] CD8.c15.ISG.IFIT1 -> CD8_ISG
[16/17] CD8.c16.MAIT.SLC4A10 -> CD8_MAIT
[17/17] CD8.c17.Tm.NME1 -> CD8_EffectorMemory

Head of df_metrics:
           cluster        Ground_Truth      variant  noise_fraction  \
0   CD8.c01.Tn.MAL           CD8_Naive     standard            0.86   
1   CD8.c01.Tn.MAL           CD8_Naive  filter_only            0.48   
2   CD8.c01.Tn.MAL           

In [13]:
# =============================================================================
# 8. Summary tables
# =============================================================================

# ----- top50 summary for panels b/c + Gini summaries -----
summary = (
    df_metrics
    .groupby("variant", as_index=False)[
        [
            "noise_fraction",
            "marker_recall",
            "mean_gene_gini",
            "low_gini_fraction",
            "high_gini_fraction",
        ]
    ]
    .mean()
)

summary["noise_pct"] = summary["noise_fraction"] * 100
summary["marker_recall_pct"] = summary["marker_recall"] * 100
summary["low_gini_pct"] = summary["low_gini_fraction"] * 100
summary["high_gini_pct"] = summary["high_gini_fraction"] * 100

summary.to_csv(OUTPUT_DIR / "fig2_cd8_backend_free_summary.csv", index=False)

print("\nTop50 summary:")
print(summary)

# ----- top-N curve summary for panel d + optional Gini curves -----
topn_summary = (
    df_topn
    .groupby(["variant", "n_top_eval"], as_index=False)[
        [
            "marker_recall_at_n",
            "noise_fraction_at_n",
            "mean_gene_gini_at_n",
            "low_gini_fraction_at_n",
            "high_gini_fraction_at_n",
        ]
    ]
    .mean()
)

topn_summary["marker_recall_pct"] = topn_summary["marker_recall_at_n"] * 100
topn_summary["noise_pct"] = topn_summary["noise_fraction_at_n"] * 100
topn_summary["low_gini_pct"] = topn_summary["low_gini_fraction_at_n"] * 100
topn_summary["high_gini_pct"] = topn_summary["high_gini_fraction_at_n"] * 100

topn_summary.to_csv(OUTPUT_DIR / "Fig2a_c_data.csv", index=False)

print("\nTop-N summary:")
print(topn_summary)


Top50 summary:
       variant  noise_fraction  marker_recall  mean_gene_gini  \
0  filter_only        0.085042       0.529412        0.586759   
1    full_core        0.000000       0.536765        0.684021   
2   regex_mask        0.007059       0.536765        0.601826   
3     standard        0.191765       0.500000        0.482145   

   low_gini_fraction  high_gini_fraction  noise_pct  marker_recall_pct  \
0           0.015294                 0.0   8.504202          52.941176   
1           0.001176                 0.0   0.000000          53.676471   
2           0.008235                 0.0   0.705882          53.676471   
3           0.183529                 0.0  19.176471          50.000000   

   low_gini_pct  high_gini_pct  
0      1.529412            0.0  
1      0.117647            0.0  
2      0.823529            0.0  
3     18.352941            0.0  

Top-N summary:
        variant  n_top_eval  marker_recall_at_n  noise_fraction_at_n  \
0   filter_only          10       

In [14]:
# =============================================================================
# 9. Fig. 1 problem schematic:
# representative dirty cluster for main Fig. 1,
# cluster-level burden distribution for Supplementary
# =============================================================================
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

fig1_cluster_list = sorted(adata.obs["meta.cluster"].astype(str).unique())
fig1_cluster_list = [c for c in fig1_cluster_list if get_cd8_ground_truth(c) not in EXCLUDE_GT]

problem_rows = []
problem_gene_rows = []
cluster_burden_rows = []

for i, cluster_name in enumerate(fig1_cluster_list, start=1):
    gt = get_cd8_ground_truth(cluster_name)
    print(f"[Fig1 problem {i}/{len(fig1_cluster_list)}] {cluster_name} -> {gt}")

    de_df_raw, _ = build_de_table(adata, cluster_name)

    genes_by_n = {}
    total_non_identity_by_n = {}

    for n_top in FIG1_TOPN_GRID:
        genes_std = get_standard_hvg_deg_genes(de_df_raw, adata, n_top=n_top)
        genes_by_n[n_top] = genes_std

        gene_categories = [
            classify_non_identity_category_for_fig1(
                g,
                gene_stats=GLOBAL_GENE_STATS,
                low_gini_thr=LOW_GINI_THR,
            )
            for g in genes_std
        ]

        total_non_identity_frac = np.mean([c != "Non-noise" for c in gene_categories]) if len(gene_categories) > 0 else np.nan
        total_non_identity_by_n[n_top] = total_non_identity_frac

        vc = pd.Series(gene_categories).value_counts()

        for cat in FIG1_NONIDENTITY_CATEGORY_ORDER:
            count_cat = int(vc.get(cat, 0))
            frac_cat = count_cat / len(genes_std) if len(genes_std) > 0 else np.nan
            problem_rows.append({
                "cluster": cluster_name,
                "Ground_Truth": gt,
                "n_top": n_top,
                "non_identity_category": cat,
                "count": count_cat,
                "fraction": frac_cat,
            })

        for rank, (g, cat) in enumerate(zip(genes_std, gene_categories), start=1):
            problem_gene_rows.append({
                "cluster": cluster_name,
                "Ground_Truth": gt,
                "n_top": n_top,
                "rank": rank,
                "gene": g,
                "non_identity_category": cat,
                "is_non_identity": cat != "Non-noise",
            })

    genes_20 = genes_by_n.get(20, [])
    genes_50 = genes_by_n.get(FIG1_SELECT_N, [])

    burden_20 = total_non_identity_by_n.get(20, np.nan)
    burden_50 = total_non_identity_by_n.get(FIG1_SELECT_N, np.nan)
    recall_50 = marker_recall_at_n(genes_50, gt, adata.var_names)

    # objective score:
    # prioritize burden at top-50, then top-20, then lower canonical recall
    problem_score = (
        (0 if pd.isna(burden_50) else burden_50) +
        0.25 * (0 if pd.isna(burden_20) else burden_20) -
        0.10 * (0 if pd.isna(recall_50) else recall_50)
    )

    cluster_burden_rows.append({
        "cluster": cluster_name,
        "Ground_Truth": gt,
        "non_identity_fraction_at_20": burden_20,
        "non_identity_fraction_at_50": burden_50,
        "marker_recall_at_50": recall_50,
        "problem_score": problem_score,
    })

df_problem = pd.DataFrame(problem_rows)
df_problem_genes = pd.DataFrame(problem_gene_rows)
df_problem_clusters = pd.DataFrame(cluster_burden_rows)

# objective selection, with optional manual override
if FORCE_FIG1_PROBLEM_CLUSTER is not None and FORCE_FIG1_PROBLEM_CLUSTER in set(df_problem_clusters["cluster"]):
    fig1_problem_cluster = FORCE_FIG1_PROBLEM_CLUSTER
else:
    fig1_problem_cluster = (
        df_problem_clusters
        .sort_values(
            ["problem_score", "non_identity_fraction_at_50", "non_identity_fraction_at_20", "marker_recall_at_50"],
            ascending=[False, False, False, True]
        )
        .iloc[0]["cluster"]
    )

print("\nSelected Fig.1 problem cluster:", fig1_problem_cluster)

# save tables
df_problem.to_csv(OUTPUT_DIR / "fig1_problem_non_identity_by_cluster_topn.csv", index=False)
df_problem_genes.to_csv(OUTPUT_DIR / "fig1_problem_non_identity_gene_categories.csv", index=False)
df_problem_clusters.sort_values("problem_score", ascending=False).to_csv(
    OUTPUT_DIR / "fig1_problem_cluster_ranking.csv", index=False
)

print("\nTop problem clusters:")
print(
    df_problem_clusters
    .sort_values("problem_score", ascending=False)
    .head(10)
)

# -------------------------------------------------------------------------
# Fig.1b representative stacked-bar panel: selected dirty cluster
# -------------------------------------------------------------------------
plot_problem_main = (
    df_problem[df_problem["cluster"] == fig1_problem_cluster]
    .pivot(index="n_top", columns="non_identity_category", values="fraction")
    .fillna(0)
)

for cat in FIG1_NONIDENTITY_CATEGORY_ORDER:
    if cat not in plot_problem_main.columns:
        plot_problem_main[cat] = 0.0
plot_problem_main = plot_problem_main[FIG1_NONIDENTITY_CATEGORY_ORDER]
plot_problem_main = plot_problem_main.loc[FIG1_TOPN_GRID]

fig, ax = plt.subplots(figsize=(5.8, 3.8), dpi=300)

x = np.arange(len(plot_problem_main.index))
bottom = np.zeros(len(plot_problem_main.index))

for cat in FIG1_NONIDENTITY_CATEGORY_ORDER:
    vals = plot_problem_main[cat].values
    ax.bar(
        x,
        vals,
        bottom=bottom,
        color=FIG1_NONIDENTITY_COLOR_MAP[cat],
        edgecolor="white",
        linewidth=0.6,
        width=0.74,
        label=cat,
    )
    bottom += vals

for i, total in enumerate(bottom):
    ax.text(
        x[i],
        total + 0.006,
        f"{total*100:.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax.set_xticks(x)
ax.set_xticklabels([str(v) for v in plot_problem_main.index])
ax.set_xlabel(f"Top N genes ({fig1_problem_cluster})", fontsize=10, fontweight="bold")
ax.set_ylabel("Fraction non-identity burden", fontsize=10, fontweight="bold")
ax.set_title(f"Representative non-identity burden", fontsize=10, fontweight="bold")

ymax = max(bottom.max() * 1.22, 0.18)
ax.set_ylim(0, ymax)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6.8,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.02),
)

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / f"Fig1b_{fig1_problem_cluster}.png",
    dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    OUTPUT_DIR / f"Fig1b_{fig1_problem_cluster}.pdf",
    bbox_inches="tight"
)
plt.show()

# -------------------------------------------------------------------------
# FigS1a: sorted cluster-level burden at top-50
# -------------------------------------------------------------------------
supp_rank = (
    df_problem_clusters
    .sort_values("non_identity_fraction_at_50", ascending=True)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.2, 5.6), dpi=300)

bar_colors = [
    CLUSTER_COLOR_MAP.get(c, "#BDBDBD")
    for c in supp_rank["cluster"]
]

bars = ax.barh(
    supp_rank["cluster"],
    supp_rank["non_identity_fraction_at_50"] * 100,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.5,
)


ax.set_xlabel("Non-identity burden in standard top-50 input (%)", fontsize=10, fontweight="bold")
ax.set_ylabel("Cluster-level burden distribution", fontsize=10, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "FigS1a.png",
    dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    OUTPUT_DIR / "FigS1a.pdf",
    bbox_inches="tight"
)
plt.show()

# -------------------------------------------------------------------------
# FigS1c: distribution across clusters for each top-N
# -------------------------------------------------------------------------
supp_dist = (
    df_problem
    .groupby(["cluster", "n_top"], as_index=False)["fraction"]
    .sum()
    .rename(columns={"fraction": "total_non_identity_fraction"})
)

fig, ax = plt.subplots(figsize=(6.6, 4.4), dpi=300)

positions = np.arange(len(FIG1_TOPN_GRID))
box_data = [
    supp_dist.loc[supp_dist["n_top"] == n, "total_non_identity_fraction"].dropna().values * 100
    for n in FIG1_TOPN_GRID
]

bp = ax.boxplot(
    box_data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showfliers=False,
)

# keep boxes neutral so the colored dots stand out
for patch in bp["boxes"]:
    patch.set_facecolor("#F0F0F0")
    patch.set_edgecolor("black")
    patch.set_linewidth(0.8)

for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.2)

for whisker in bp["whiskers"]:
    whisker.set_color("black")
    whisker.set_linewidth(0.8)

for cap in bp["caps"]:
    cap.set_color("black")
    cap.set_linewidth(0.8)

# overlay jittered points with cluster-specific colors
rng = np.random.default_rng(RANDOM_SEED)
for i, n in enumerate(FIG1_TOPN_GRID):
    sub = supp_dist[supp_dist["n_top"] == n].copy()
    xj = rng.normal(loc=positions[i], scale=0.05, size=len(sub))

    point_colors = [CLUSTER_COLOR_MAP.get(c, "#9E9E9E") for c in sub["cluster"]]

    ax.scatter(
        xj,
        sub["total_non_identity_fraction"] * 100,
        s=22,
        c=point_colors,
        alpha=0.9,
        edgecolors="white",
        linewidths=0.3,
        zorder=3,
    )

ax.set_xticks(positions)
ax.set_xticklabels([str(n) for n in FIG1_TOPN_GRID])
ax.set_xlabel("Top N genes (input size)", fontsize=10, fontweight="bold")
ax.set_ylabel("Cluster-level non-identity burden (%)", fontsize=10, fontweight="bold")
# ax.set_title("Distribution across CD8 clusters (Supplementary)", fontsize=10, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "FigS1c.png",
    dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    OUTPUT_DIR / "FigS1c.pdf",
    bbox_inches="tight"
)
plt.show()


[Fig1 problem 1/17] CD8.c01.Tn.MAL -> CD8_Naive
[Fig1 problem 2/17] CD8.c02.Tm.IL7R -> CD8_EffectorMemory
[Fig1 problem 3/17] CD8.c03.Tm.RPS12 -> CD8_EffectorMemory
[Fig1 problem 4/17] CD8.c04.Tm.CD52 -> CD8_EffectorMemory
[Fig1 problem 5/17] CD8.c05.Tem.CXCR5 -> CD8_EffectorMemory
[Fig1 problem 6/17] CD8.c06.Tem.GZMK -> CD8_EffectorMemory
[Fig1 problem 7/17] CD8.c07.Temra.CX3CR1 -> CD8_Effector
[Fig1 problem 8/17] CD8.c08.Tk.TYROBP -> CD8_Effector
[Fig1 problem 9/17] CD8.c09.Tk.KIR2DL4 -> CD8_Effector
[Fig1 problem 10/17] CD8.c10.Trm.ZNF683 -> CD8_EffectorMemory
[Fig1 problem 11/17] CD8.c11.Tex.PDCD1 -> CD8_Exhausted
[Fig1 problem 12/17] CD8.c12.Tex.CXCL13 -> CD8_Exhausted
[Fig1 problem 13/17] CD8.c13.Tex.myl12a -> CD8_Exhausted
[Fig1 problem 14/17] CD8.c14.Tex.TCF7 -> CD8_Exhausted
[Fig1 problem 15/17] CD8.c15.ISG.IFIT1 -> CD8_ISG
[Fig1 problem 16/17] CD8.c16.MAIT.SLC4A10 -> CD8_MAIT
[Fig1 problem 17/17] CD8.c17.Tm.NME1 -> CD8_EffectorMemory

Selected Fig.1 problem cluster: CD8.c01.T

In [15]:
# =============================================================================
# 11. Representative rank-shift cluster
# =============================================================================
rep_cluster = FIG1_GINI_CLUSTER
if rep_cluster not in set(adata.obs["meta.cluster"].astype(str)):
    rep_cluster = cluster_list[0]

rep_raw, rep_filtered = build_de_table(adata, rep_cluster)
genes_std_50 = get_standard_hvg_deg_genes(rep_raw, adata, n_top=50)
genes_core_50 = get_full_core_genes(curator, adata, rep_cluster, n_top=50)

rank_std = {g: i + 1 for i, g in enumerate(genes_std_50)}
rank_core = {g: i + 1 for i, g in enumerate(genes_core_50)}

genes_union = []
for g in genes_std_50[:15] + genes_core_50[:15]:
    if g not in genes_union:
        genes_union.append(g)

OFFSCALE_RANK = 55

plot_df = pd.DataFrame({
    "gene": genes_union,
    "standard_rank": [rank_std.get(g, OFFSCALE_RANK) for g in genes_union],
    "full_core_rank": [rank_core.get(g, OFFSCALE_RANK) for g in genes_union],
})

plot_df.to_csv(OUTPUT_DIR / "Fig2d_data.csv", index=False)

print("\nRepresentative rank-shift data:")
print(plot_df)


Representative rank-shift data:
      gene  standard_rank  full_core_rank
0     IL7R              1               1
1     CCR7              2               2
2     KLF2              3               3
3    RPLP0              4              55
4     RPS2              5              55
5     RPSA              6              55
6      LTB              7               4
7     JUNB              8              55
8     LEF1              9               5
9     FTH1             10              55
10     FOS             11              55
11    SELL             12               6
12    LDHB             13               7
13  SLC2A3             14               8
14  GPR183             15               9
15    TCF7             16              10
16   S1PR1             17              11
17   NELL2             18              12
18    NPM1             19              13
19    KLF3             20              14
20    TOB1             21              15


In [16]:
# =============================================================================
# 12. Plot updated backend-free minimal Fig.2
#    a: biological-noise fraction at top50
#    b: low-Gini housekeeping fraction at top50
#    c: canonical marker recall across top N
#      + thin auxiliary dashed lines for noise fraction
#    d: representative rank shift
# =============================================================================
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

order = ["standard", "filter_only", "regex_mask", "full_core"]

display_map = {
    "standard": "standard",
    "filter_only": "filter_only",
    "regex_mask": "regex_mask",
    "full_core": "full_core",
}


color_map = {
    "standard": "#5DA5DA",
    "filter_only": "#F0B43C",
    "regex_mask": "#60BD68",
    "full_core": "#B276B2",
}

summary_idx = summary.set_index("variant")

# safety: compute noise_pct in topn_summary if not present
if "noise_pct" not in topn_summary.columns:
    topn_summary = topn_summary.copy()
    topn_summary["noise_pct"] = topn_summary["noise_fraction_at_n"] * 100

def nice_ylim(values, pad_frac=0.12, min_pad=1.0, floor_zero=True):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (0, 1)
    vmin = vals.min()
    vmax = vals.max()
    pad = max((vmax - vmin) * pad_frac, min_pad)
    ymin = 0 if floor_zero else vmin - pad
    ymax = vmax + pad
    return ymin, ymax

fig = plt.figure(figsize=(13, 8), dpi=250)

# -------------------------------------------------------------------------
# Panel a: biological-noise fraction in top-50 genes
# -------------------------------------------------------------------------
ax1 = plt.subplot2grid((2, 2), (0, 0))

vals_b = [summary_idx.loc[v, "noise_pct"] for v in order]
ax1.bar(
    [display_map[v] for v in order],
    vals_b,
    color=[color_map[v] for v in order],
    edgecolor="black",
    linewidth=1.2,
)

for i, v in enumerate(vals_b):
    ax1.text(i, v + 0.2, f"{v:.1f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax1.set_ylabel("Biological-noise fraction (%)", fontweight="bold", fontsize=13)
ax1.set_xlabel("Biological-noise fraction in top-50 genes", fontweight="bold", fontsize=13)
ax1.set_ylim(*nice_ylim(vals_b, pad_frac=0.10, min_pad=1.2))
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# -------------------------------------------------------------------------
# Panel b: low-Gini housekeeping fraction in top-50 genes
# -------------------------------------------------------------------------
ax2 = plt.subplot2grid((2, 2), (0, 1))

vals_c = [summary_idx.loc[v, "low_gini_pct"] for v in order]
ax2.bar(
    [display_map[v] for v in order],
    vals_c,
    color=[color_map[v] for v in order],
    edgecolor="black",
    linewidth=1.2,
)

for i, v in enumerate(vals_c):
    ax2.text(i, v + 0.2, f"{v:.1f}", ha="center", va="bottom", fontsize=13, fontweight="bold")

ax2.set_ylabel("Low-Gini fraction (%)", fontweight="bold", fontsize=13)
ax2.set_xlabel("Low-Gini housekeeping fraction in top-50 genes", fontweight="bold", fontsize=13)
ax2.set_ylim(*nice_ylim(vals_c, pad_frac=0.10, min_pad=1.0))
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# -------------------------------------------------------------------------
# Panel c: canonical marker recall across top N
#           + thin auxiliary dashed lines for noise fraction
# -------------------------------------------------------------------------
ax3 = plt.subplot2grid((2, 2), (1, 0))
ax3b = ax3.twinx()

# main recall lines
for variant in order:
    sub = topn_summary[topn_summary["variant"] == variant].sort_values("n_top_eval")
    ax3.plot(
        sub["n_top_eval"],
        sub["marker_recall_pct"],
        marker="o",
        linewidth=2.4,
        markersize=6,
        color=color_map[variant],
        label=display_map[variant],
        zorder=3,
    )

# thin auxiliary noise lines
for variant in order:
    sub = topn_summary[topn_summary["variant"] == variant].sort_values("n_top_eval")
    ax3b.plot(
        sub["n_top_eval"],
        sub["noise_pct"],
        linestyle="--",
        linewidth=1.2,
        alpha=0.55,
        color=color_map[variant],
        zorder=1,
    )

all_recall_vals = topn_summary["marker_recall_pct"].values
ymin_d, ymax_d = nice_ylim(all_recall_vals, pad_frac=0.10, min_pad=1.0, floor_zero=False)

all_noise_vals = topn_summary["noise_pct"].values
_, ymax_noise = nice_ylim(all_noise_vals, pad_frac=0.15, min_pad=1.5, floor_zero=True)

ax3.set_ylabel("Canonical marker recall (%)", fontweight="bold", fontsize=13)
ax3b.set_ylabel("Biological-noise fraction (%)", color="dimgray", fontweight="bold", fontsize=13)
ax3.set_xlabel("Top N genes", fontweight="bold", fontsize=13)

ax3.set_xticks(TOPN_GRID)
ax3.set_ylim(ymin_d, ymax_d)
ax3b.set_ylim(0, ymax_noise)

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3b.spines["top"].set_visible(False)

ax3.tick_params(axis="y")
ax3b.tick_params(axis="y", colors="dimgray")

# legend: variants + style explanation
variant_handles = [
    Line2D([0], [0], color=color_map[v], lw=2.4, marker="o", label=display_map[v])
    for v in order
]
style_handles = [
    Line2D([0], [0], color="black", lw=2.4, linestyle="-", label="recall"),
    Line2D([0], [0], color="black", lw=1.2, linestyle="--", alpha=0.45, label="noise"),
]

ax3.legend(
    handles=variant_handles + style_handles,
    frameon=False,
    fontsize=10,
    loc="upper left",
    ncol=4,
)


# -------------------------------------------------------------------------
# Panel d: representative rank shift
# -------------------------------------------------------------------------
ax4 = plt.subplot2grid((2, 2), (1, 1))

gt_rep = get_cd8_ground_truth(rep_cluster)

rank_label_ceiling = 18
rank_plot_ceiling = 20


x_left_label = -0.02
x_right_label = 1.02

for _, row in plot_df.iterrows():
    g = row["gene"]
    y1 = row["standard_rank"]
    y2 = row["full_core_rank"]

    if g in CD8_MARKER_DB.get(gt_rep, set()):
        color = "#D32F2F"   # canonical marker
        lw = 2.2
        z = 3
    elif is_any_noise_gene(g):
        color = "#9E9E9E"   # noise
        lw = 1.2
        z = 1
    else:
        color = "#4F81BD"   # rescued / other informative
        lw = 1.2
        z = 2

    ax4.plot([0, 1], [y1, y2], color=color, linewidth=lw, alpha=0.9, zorder=z)
    ax4.scatter([0, 1], [y1, y2], color=color, s=18, zorder=z)

    # 左側ラベル
    if y1 <= rank_label_ceiling:
        ax4.text(
            x_left_label, y1, g,
            ha="right", va="center",
            fontsize=12,
            clip_on=False,
        )

    # 右側ラベル
    if y2 <= rank_label_ceiling:
        ax4.text(
            x_right_label, y2, g,
            ha="left", va="center",
            fontsize=12,
            clip_on=False,
        )

ax4.set_xticks([0, 1])
ax4.set_xticklabels(["Standard", "Full core"])

ax4.set_xlim(-0.28, 1.28)

ax4.set_ylim(rank_plot_ceiling, 0)
ax4.set_yticks(range(1, rank_plot_ceiling + 1, 2))

ax4.tick_params(axis="y", labelsize=8, pad=2)

ax4.set_ylabel("Gene rank", fontweight="bold", fontsize=13)
ax4.set_xlabel(f"Representative rank shift ({rep_cluster})", fontsize=13, fontweight="bold")

#ax4.yaxis.set_label_coords(-0.16, 0.5)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Fig2a_d.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "Fig2a_d.pdf", bbox_inches="tight")
plt.show()